# Modélisation du prix d'une pièce
## Hypothèse:
### prix = f(prix_materiau(€/kg), volume(m^3), masse volumique(kg/m^3), difficulté d'usinage)

# <span style="color:red">**I**</span> Prix du matériau

In [22]:
import pandas as pd
import os
import cadquery as cq
df = pd.DataFrame({
'matiere':
[
    '42 CrMO4+QT',
    'S275 J2+N Ceq < 0,40',
    'S275 J2+N',
    'S355J2+N Ceq < 0,43',
    'S355J2+N',
    'Inox 304L',
    'Alu',
    'S355J2'
 ],
 'prix(€/kg)-01-01-2024':
 [
2.2,
2.3,
2.9,
2,
1.9,
5,
4.6,
1.9
 ]
 })
df

,matiere,prix(€/kg)-01-01-2024
0,42 CrMO4+QT,2.2
1,"S275 J2+N Ceq < 0,40",2.3
2,S275 J2+N,2.9
3,"S355J2+N Ceq < 0,43",2.0
4,S355J2+N,1.9
5,Inox 304L,5.0
6,Alu,4.6
7,S355J2,1.9


# <span style="color:red">**II**</span> Volume et masse volumique

In [20]:
# Estimations
df['masse-volumique']=[
    7950,
    7850,
    7850,
    7850,
    7850,
    7900,
    2700,
    7850,
]
df

,matiere,prix(€/kg)-01-01-2024,masse-volumique
0,42 CrMO4+QT,2.2,7950
1,"S275 J2+N Ceq < 0,40",2.3,7850
2,S275 J2+N,2.9,7850
3,"S355J2+N Ceq < 0,43",2.0,7850
4,S355J2+N,1.9,7850
5,Inox 304L,5.0,7900
6,Alu,4.6,2700
7,S355J2,1.9,7850


# <span style="color:red">**III**</span> Difficulté d'usinage

In [21]:
dossier_stp = "../3D"
# Initialisation d'une liste pour stocker les données
data = []
# Parcourir tous les fichiers dans le dossier
for fichier in os.listdir(dossier_stp):
    if fichier.endswith(".stp") or fichier.endswith(".step"):
        path_to_stp_file = os.path.join(dossier_stp, fichier)
        try:
            # Charger le modèle 3D à partir du fichier STEP
            model = cq.importers.importStep(path_to_stp_file)
            shape = model.val()

            # Extraire les caractéristiques
            volume = shape.Volume()
            surface = shape.Area()
            bbox = shape.BoundingBox()
            longueur, largeur, hauteur = bbox.xlen, bbox.ylen, bbox.zlen
            faces = len(shape.Faces())
            aretes = len(shape.Edges())
            sommets = len(shape.Vertices())

            # Ajouter au DataFrame
            data.append({
                "Fichier": fichier,
                "Vol": volume,
                "Surf": surface,
                "Lng_BBox": longueur,
                "Lrg_BBox": largeur,
                "Ht_BBox": hauteur,
                "Faces": faces,
                "Aretes": aretes,
                "Sommets": sommets
            })
        except Exception as e:
            print(f"Erreur lors du traitement du fichier {fichier}: {e}")

# Créer un DataFrame à partir des données
df = pd.DataFrame(data)

# Nombres de faces?
df['Faces']



0     213
1     213
2     608
3     205
4     114
5     270
6      44
7      96
8     112
9     100
10    106
11    184
12     10
13      8
14      8
15     10
16    130
17    133
18     92
19     18
20     20
21     48
22      6
23      7
24      6
25     15
26     10
27     86
28     86
29    170
30    170
31     10
32    112
33     99
34     56
35     38
36     68
37    311
38     96
39    741
40    127
41     88
42    115
43    741
44    106
Name: Faces, dtype: int64

# Quelle est l'épaisseur de la tôle?
Je cherche à calculer l'épaisseur d'une pièce 3D.
Hypothèse: l'épaisseur de la tôle correspond à la plus petite valeur de distance entre les points répétée le plus grand nombre de fois.
Pour chaque point, trouve les 10 points les plus proches et stocke leur distance arrondie à l'unité dans une liste sur laquelle tu 
Utilise les données provenant de ceci:
